In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, TimestampType,
    StringType, LongType, DateType
)
from datetime import datetime
import re

def audit_table_bronze(spark):
    """
    Creates audit records for bronze layer files.
    """

    # -------------------------------
    # Schema for audit table
    # -------------------------------
    schema = StructType([
        StructField("runid", IntegerType(), True),
        StructField("load_date", TimestampType(), True),
        StructField("table_name", StringType(), True),
        StructField("count", LongType(), True),
        StructField("file_date", DateType(), True)
    ])

    # Empty dataframe to hold audit results
    audit_df = spark.createDataFrame([], schema)

    # Current timestamp
    current_timestamp = datetime.now()

    # Audit table name
    audit_table = "prod_lca_unrestricted.bassett_epic_amb_bronze.audit"

    # -------------------------------
    # Fetch max RunId from pipeline log
    # -------------------------------
    df_pipeline = spark.sql(
        """
        SELECT MAX(RunId) AS max_id
        FROM prod_ica_unrestricted.bassett_epic_amb_silver.adf_pipeline_logs
        """
    )

    max_id_row = df_pipeline.collect()[0]
    max_id = max_id_row["max_id"]

    # Initialize RunId counter
    id_counter = max_id + 1 if max_id is not None else 1

    # -------------------------------
    # File pattern
    # -------------------------------
    pattern = r".*_(DetailDictionary)_(.*?)_EPIC.*_(\d{8})\.txt"

    # File path
    base_path = "dbfs:/mnt/bassett_source_test/raw_files/ambulatory/prod/bassett/"
    files = dbutils.fs.ls(base_path)

    for f in files:

        # print(f"Processing file: {f.name}")

        match = re.match(pattern, f.name)

        if match:
            file_name = match.group(2)
            file_date_str = match.group(3)
            file_date = datetime.strptime(file_date_str, "%Y%m%d").date()
        else:
            print(f"No match found for {f.name}")
            continue

        # -------------------------------
        # Read file
        # -------------------------------
        file_path = f"{base_path}{f.name}"

        df = (
            spark.read.format("csv")
            .option("header", "true")
            .option("delimiter", "|")
            .option("inferSchema", "true")
            .load(file_path)
        )

        # -------------------------------
        # Row count
        # -------------------------------
        row_count = df.count()
        print(f"Row Count for {file_name}: {row_count}")

        # -------------------------------
        # Create audit record
        # -------------------------------
        record = [(id_counter, current_timestamp, file_name, row_count, file_date)]
        record_df = spark.createDataFrame(record, schema)

        audit_df = audit_df.union(record_df)

        id_counter += 1

    # -------------------------------
    # Write audit data to table
    # -------------------------------
    if audit_df.count() > 0:
        audit_df.write.format("delta").mode("append").saveAsTable(audit_table)
        print(f"Audit records successfully written to {audit_table}")
    else:
        print("No audit records generated.")


In [0]:
# Empty list to store temporary table names
temp_tables_list = []

# Iterate over the list of bronze tables
for table in bronze_tbl_list:

    # Dynamically construct table names
    bronze_table_name = f"prod_lca_unrestricted.bassett_epic_amb_bronze.{table}"
    silver_table_name = f"prod_ica_unrestricted.bassett_epic_amb_silver.{table}"

    try:
        # -------------------------------
        # Get max LoadDate from silver
        # -------------------------------
        silver_max_current_date_row = spark.sql(
            f"""
            SELECT MAX(LoadDate) AS max_date
            FROM {silver_table_name}
            """
        ).collect()

        silver_max_current_date = (
            silver_max_current_date_row[0]["max_date"]
            if silver_max_current_date_row
            and silver_max_current_date_row[0]["max_date"]
            else "1900-01-01 00:00:00"
        )

        print(f"-- {table} max LoadDate is: {silver_max_current_date}")

        # -------------------------------
        # Filter bronze data incrementally
        # -------------------------------
        filtered_bronze_data = spark.sql(
            f"""
            SELECT *
            FROM {bronze_table_name}
            WHERE LoadDate > '{silver_max_current_date}'
            """
        )

        # -------------------------------
        # Create temporary view
        # -------------------------------
        temp_table_name = f"{table}_temp"
        filtered_bronze_data.createOrReplaceTempView(temp_table_name)

        # -------------------------------
        # Count new records
        # -------------------------------
        count_filtered_bronze_data = filtered_bronze_data.count()
        print(
            f"Filtered Data Count for {table}: "
            f"{count_filtered_bronze_data}"
        )

        # -------------------------------
        # Store temp table name
        # -------------------------------
        temp_tables_list.append(temp_table_name)

        # print(f"Temporary Table Created for {table}: {temp_table_name}")

    except Exception as e:
        print(
            f"Error in filtering data while processing table {table}: {str(e)}"
        )

# -------------------------------
# Final list of temp tables
# -------------------------------
# print("List of Temp Tables:", temp_tables_list)


In [0]:
from delta.tables import DeltaTable

def restore_table(spark, table_name):
    """
    Restores a Delta table to its previous version.
    """

    try:
        # Load Delta table
        delta_table = DeltaTable.forName(spark, table_name)

        # Get history
        history_df = delta_table.history()

        # Get latest version
        latest_version = (
            history_df.select("version")
                      .agg({"version": "max"})
                      .collect()[0][0]
        )

        if latest_version > 0:
            target_version = latest_version - 1

            restore_query = (
                f"RESTORE TABLE {table_name} "
                f"VERSION AS OF {target_version}"
            )

            spark.sql(restore_query)

            print(
                f"Table '{table_name}' successfully restored "
                f"to version {target_version}"
            )
        else:
            print(
                f"Table '{table_name}' is already at the earliest version. "
                f"Nothing to restore."
            )

    except Exception as e:
        print(f"Error restoring table '{table_name}': {str(e)}")


In [0]:
from pyspark.sql.functions import col, expr, max
import os

def check_duplicate(spark, table_name, primary_keys, base_log_dir_path):
    """
    Checks for duplicate records based on primary keys and logs errors if found.
    """

    # Read table
    df = spark.sql(f"SELECT * FROM {table_name}")

    validation_issues = []

    # -------------------------
    # Find duplicate keys
    # -------------------------
    duplicate_keys = (
        df.groupBy(*primary_keys)
          .agg(expr("count(*)").alias("duplicate_count"))
          .filter(col("duplicate_count") > 1)
    )

    if duplicate_keys.count() > 0:

        # Get full duplicate rows
        duplicate_rows = df.join(
            duplicate_keys.select(*primary_keys),
            on=primary_keys,
            how="inner"
        )

        # Get max FileDate and count per PK
        duplicate_with_max_date = (
            duplicate_rows
            .groupBy(*primary_keys)
            .agg(
                max("FileDate").alias("max_load_date"),
                expr("count(*)").alias("duplicate_count")
            )
        )

        duplicate_records = duplicate_with_max_date.collect()

        for record in duplicate_records:
            # Primary key values
            pk_values = {key: record[key] for key in primary_keys}

            max_load_date = record["max_load_date"]
            duplicate_count = record["duplicate_count"]

            error_message = (
                f"Fail: Duplicate records found in table '{table_name}'. "
                f"Count: {duplicate_count} for primary key {pk_values} "
                f"for load date {max_load_date}"
            )

            validation_issues.append(error_message + "\n")

            # -------------------------
            # Create log folder
            # -------------------------
            date_only = str(max_load_date)[:10]

            current_date_folder_path = os.path.join(
                base_log_dir_path, date_only
            )

            dbutils.fs.mkdirs(current_date_folder_path)

            # Log file path
            log_file_path = os.path.join(
                current_date_folder_path,
                f"{date_only}_{table_name}_duplicate_error_log.txt"
            )

            # Write log
            error_list = "".join(validation_issues)
            dbutils.fs.put(log_file_path, error_list, overwrite=True)

    else:
        print(f"Pass: Duplicate check for table {table_name}")

    return validation_issues


In [0]:
from pyspark.sql.functions import col
import os

def check_null(spark, table_name, primary_keys, base_log_dir_path):
    """
    Checks for NULL values in primary key columns and logs errors if found.
    """

    # Read table
    df = spark.sql(f"SELECT * FROM {table_name}")

    validation_issues = []

    for column_name in primary_keys:
        # Find rows where PK column is NULL
        null_rows = df.filter(col(column_name).isNull()) \
                      .select("FileDate", column_name)

        null_count = null_rows.count()

        if null_count > 0:
            # Collect load dates
            load_dates = null_rows.select("FileDate").distinct().collect()
            load_date_str = ",".join([str(row["FileDate"]) for row in load_dates])

            error_message = (
                f"Fail: Null values found in primary key column '{column_name}' "
                f"for table '{table_name}'. "
                f"Count: {null_count} for load dates {load_date_str}"
            )

            validation_issues.append(error_message + "\n")

            # -------------------------
            # Create log folder
            # -------------------------
            for row in load_dates:
                date_only = str(row["FileDate"])[:10]

                current_date_folder_path = os.path.join(
                    base_log_dir_path, date_only
                )

                dbutils.fs.mkdirs(current_date_folder_path)

                # Log file path
                log_file_path = os.path.join(
                    current_date_folder_path,
                    f"{date_only}_{table_name}_null_error_log.txt"
                )

                # Write log
                error_list = "".join(validation_issues)
                dbutils.fs.put(log_file_path, error_list, overwrite=True)

    # -------------------------
    # Final status
    # -------------------------
    if not validation_issues:
        print(f"Pass: Null check for table {table_name}")

    return validation_issues


In [0]:
from pyspark.sql.functions import lit
from delta.tables import DeltaTable

if validation_success:

    # -------------------------
    # UPSERT / APPEND / RESET
    # -------------------------
    for table in temp_tables_list:

        # Read dataframe from temp table
        df = spark.sql(f"SELECT * FROM {table}")

        # Add client id column
        df = df.withColumn("ica_client_id", lit(5).cast("string"))

        # Sanitize column names
        df = sanitize_column_names(df)

        # -------------------------
        # Target Silver Table
        # -------------------------
        silver_table_data = DeltaTable.forName(
            spark,
            f"prod_ica_unrestricted.bassett_epic_amb_silver.{table}"
        )

        # Primary keys
        primary_keys = table_metadata[table]["PrimaryKey"]

        # Merge join condition
        join_condition = " AND ".join(
            [f"target.{key} = source.{key}" for key in primary_keys]
        )

        # -------------------------
        # UPSERT / DROP_INSERT
        # -------------------------
        if table_metadata[table]["type"] in ["UPSERT", "DROP_INSERT"]:

            (
                silver_table_data.alias("target")
                .merge(df.alias("source"), join_condition)
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )

            print(f"- {table} table upserted to silver successfully")

        # -------------------------
        # OVERWRITE
        # -------------------------
        else:
            df.write.mode("overwrite").saveAsTable(
                f"prod_ica_unrestricted.bassett_epic_amb_silver.{table}"
            )
            print(f"- {table} table overwritten to silver successfully")

# =========================================================
# VALIDATION FAILED → RESTORE TABLES
# =========================================================
else:
    print("Validation failed, restoring all tables....")

    for table in bronze_tbl_list:
        restore_table_data = (
            f"prod_ica_unrestricted.bassett_epic_amb_bronze.{table}"
        )

        try:
            if table in [
                "ChargeHold",
                "FBNS",
                "PatientReferrals",
                "PatientVerification"
            ]:
                # Restoring DROP & INSERT tables
                restore_table_drop_insert(spark, restore_table_data)
            else:
                restore_table(spark, restore_table_data)

            print(f"Successfully {table} restored")

        except ValueError as e:
            print(f"Error while restoring table {table}, Error: {str(e)}")

    print("Processing complete. Check log files.")
